# 06 - IEEE-CIS Logical Rule Ablation

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys

KAGGLE = Path("/kaggle").exists()
REPO_URL = "https://github.com/Tommyhuy1705/Explainable_NeuroSymbolic_Fraud_Detection.git"
KAGGLE_PROJECT_DIR = Path("/kaggle/working/Explainable_NeuroSymbolic_Fraud_Detection")

if KAGGLE:
    os.environ.setdefault("THESIS_QUICK_RUN", "0")
    os.environ.setdefault("THESIS_SYNTHETIC_FALLBACK", "0")

def find_project_root() -> Path | None:
    direct_candidates = [KAGGLE_PROJECT_DIR, Path.cwd(), *Path.cwd().parents]
    for candidate in direct_candidates:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate
    for base in (Path("/kaggle/working"), Path("/kaggle/input")):
        if base.exists():
            matches = sorted(path.parent for path in base.glob("**/configs") if path.is_dir())
            for candidate in matches:
                if (candidate / "src").is_dir():
                    return candidate
    return None

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None and KAGGLE:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(KAGGLE_PROJECT_DIR)],
        check=True,
    )
    PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root with src/ and configs/ was not found")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

AUDIT_SOURCE_FILES = (
    "scripts/generate_notebooks.py",
    "src/artifacts.py",
    "src/data/dataset.py",
    "src/data/preprocessing.py",
    "src/experiment.py",
    "src/explanation/explanation_metrics.py",
    "src/explanation/rule_explainer.py",
    "src/logic/fraud_rules.py",
    "src/logic/knowledge_base.py",
    "src/logic/predicates.py",
    "src/logic/tensor_logic.py",
)

def audit_pipeline_fingerprint(config_path: Path) -> str:
    paths = [PROJECT_ROOT / relative for relative in AUDIT_SOURCE_FILES]
    paths.append(Path(config_path))
    missing = [str(path) for path in paths if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"Files required for the audit-pipeline fingerprint are missing: {missing}")
    digest = hashlib.sha256()
    for path in sorted(paths, key=lambda item: item.relative_to(PROJECT_ROOT).as_posix()):
        relative = path.relative_to(PROJECT_ROOT).as_posix()
        digest.update(relative.encode("utf-8"))
        digest.update(b"\0")
        digest.update(path.read_bytes())
        digest.update(b"\0")
    return digest.hexdigest()

QUICK_RUN = os.getenv("THESIS_QUICK_RUN", "0") == "1"
ALLOW_SYNTHETIC_FALLBACK = os.getenv("THESIS_SYNTHETIC_FALLBACK", "0") == "1"
OUTPUT_BASE = Path("/kaggle/working/thesis_outputs") if KAGGLE else PROJECT_ROOT / "results/runs/notebooks"
INPUT_ROOTS = [OUTPUT_BASE, PROJECT_ROOT / "results/runs/notebooks"]
if Path("/kaggle/input").exists():
    INPUT_ROOTS.append(Path("/kaggle/input"))

try:
    GIT_COMMIT = subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], text=True
    ).strip()
except (OSError, subprocess.CalledProcessError):
    GIT_COMMIT = None

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
print({
    "project_root": str(PROJECT_ROOT),
    "git_commit": GIT_COMMIT,
    "quick_run": QUICK_RUN,
    "synthetic_fallback": ALLOW_SYNTHETIC_FALLBACK,
    "kaggle": KAGGLE,
})

## Thiết lập

Ablation giữ nguyên frozen IEEE predictor, calibrated probabilities và threshold.
Chỉ rule subset thay đổi giữa các điều kiện. Đây là post-hoc diagnostic trên locked test,
không phải bước chọn một final rule set mới.

In [ ]:
preflight_manifests = []
preflight_artifacts = []
for root_value in INPUT_ROOTS:
    root = Path(root_value)
    if not root.exists():
        continue
    manifest_paths = [root] if root.is_file() and root.name == "frozen_reference_manifest.json" else list(root.glob("**/frozen_reference_manifest.json"))
    for manifest_path in manifest_paths:
        try:
            manifest_payload = json.loads(manifest_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        if str(manifest_payload.get("dataset_name", "")).lower() != "ieee_cis".lower():
            continue
        preflight_manifests.append(manifest_path.resolve())
        artifact_path = manifest_path.parent / str(manifest_payload.get("artifact_file", ""))
        if artifact_path.exists():
            preflight_artifacts.append(artifact_path.resolve())

preflight_table = pd.DataFrame({
    "manifest": [str(path) for path in sorted(set(preflight_manifests))],
})
display(preflight_table)
print("Frozen artifacts:")
for path in sorted(set(preflight_artifacts)):
    print(path)
if not preflight_manifests or not preflight_artifacts:
    raise FileNotFoundError(
        "No complete ieee_cis frozen artifact was found below INPUT_ROOTS. "
        "On Kaggle, attach the corresponding benchmark notebook output; locally, place it below results/runs/notebooks."
    )

In [ ]:
from src.artifacts import assert_frozen_alignment, load_frozen_reference_artifact, sha256_file
from src.data import load_config, prepare_dataset
from src.experiment import load_experiment_data

config = load_config(PROJECT_ROOT / "configs/ieee_cis.yaml")
frame, data_source = load_experiment_data(
    config, max_rows=12000 if QUICK_RUN else None,
    synthetic_fallback=ALLOW_SYNTHETIC_FALLBACK,
    synthetic_rows=12000 if QUICK_RUN else 6000,
)
prepared = prepare_dataset(frame, config)
artifact = load_frozen_reference_artifact(
    "ieee_cis", expected_config=config,
    search_roots=[OUTPUT_BASE / "02_ieee_cis_model_benchmarks", *INPUT_ROOTS],
)
assert_frozen_alignment(artifact, prepared.y_validation, prepared.y_test)
if bool(artifact["manifest"]["quick_run"]) != QUICK_RUN:
    raise ValueError("Notebook mode and frozen artifact quick_run flag do not match")
if int(artifact["manifest"]["reference_seed"]) != int(config["evaluation"]["reference_seed"]):
    raise ValueError("Frozen artifact reference seed does not match the locked dataset protocol")
if not QUICK_RUN and str(artifact["manifest"].get("data_source", "")).lower() == "synthetic":
    raise ValueError("Full thesis evaluation cannot consume a synthetic-fallback frozen artifact")
probabilities = artifact["test_probability"]
threshold = float(artifact["manifest"]["threshold"])
print({
    "data_source": data_source,
    "reference_model": artifact["manifest"]["model"],
    "reference_seed": artifact["manifest"]["reference_seed"],
    "calibration_method": artifact["manifest"]["calibration_method"],
    "threshold": threshold,
    "artifact": str(artifact["artifact_path"]),
})

def write_upstream_lineage(destination, notebook_id, output_files, config_path):
    output_files = list(output_files)
    lineage = {
        "notebook_id": notebook_id,
        "git_commit": GIT_COMMIT,
        "dataset_name": artifact["manifest"]["dataset_name"],
        "data_source": data_source,
        "frozen_data_source": artifact["manifest"].get("data_source"),
        "quick_run": QUICK_RUN,
        "reference_model_key": artifact["manifest"]["model_key"],
        "reference_seed": artifact["manifest"]["reference_seed"],
        "config_sha256": artifact["manifest"]["config_sha256"],
        "audit_source_sha256": audit_pipeline_fingerprint(config_path),
        "frozen_manifest_sha256": sha256_file(artifact["manifest_path"]),
        "frozen_artifact_sha256": sha256_file(artifact["artifact_path"]),
        "output_files": output_files,
        "output_sha256": {
            name: sha256_file(Path(destination) / name) for name in output_files
        },
    }
    lineage_path = Path(destination) / "upstream_lineage.json"
    lineage_path.write_text(json.dumps(lineage, indent=2), encoding="utf-8")
    return lineage_path

In [ ]:
from src.logic import FraudRuleEngine

output_dir = OUTPUT_BASE / "06_ieee_cis_rule_ablation"
output_dir.mkdir(parents=True, exist_ok=True)
target = config["dataset"]["target_column"]
engine = FraudRuleEngine(config["logic"]["rules"]).fit(prepared.train_frame, target)
truth = engine.evaluate(prepared.test_frame)
predicted_alert = probabilities >= threshold
activation = float(config["logic"]["activation_threshold"])

def score_subset(name, columns, activation_threshold=activation):
    rule_score = truth[columns].max(axis=1).to_numpy(float) if columns else np.zeros(len(truth))
    explained = rule_score >= activation_threshold
    explained_alert = explained & predicted_alert
    evidence_without_alert = explained & ~predicted_alert
    alert_count = int(predicted_alert.sum())
    non_alert_count = int((~predicted_alert).sum())
    explained_count = int(explained.sum())
    explained_alert_count = int(explained_alert.sum())
    explained_alert_fraud_count = int(prepared.y_test[explained_alert].sum())
    base_precision = float(prepared.y_test[predicted_alert].mean()) if alert_count else np.nan
    explained_precision = (
        float(prepared.y_test[explained_alert].mean()) if explained_alert_count else np.nan
    )
    alert_support_rate = explained_alert_count / alert_count if alert_count else np.nan
    non_alert_no_evidence_rate = (
        float((~explained & ~predicted_alert).sum()) / non_alert_count
        if non_alert_count else np.nan
    )
    return {
        "ablation": name, "activation_threshold": activation_threshold, "rule_count": len(columns),
        "analysis_role": "post_hoc_locked_test_diagnostic",
        "test_rows": len(truth),
        "predicted_alert_count": alert_count,
        "explained_count": explained_count,
        "explained_alert_count": explained_alert_count,
        "explained_alert_fraud_count": explained_alert_fraud_count,
        "coverage_all": float(explained.mean()),
        "coverage_alerts": alert_support_rate,
        "unsupported_alert_rate": 1.0 - alert_support_rate if alert_count else np.nan,
        "rule_evidence_without_alert_rate": (
            float(evidence_without_alert.sum()) / non_alert_count if non_alert_count else np.nan
        ),
        "explained_alert_precision": explained_precision,
        "all_alert_precision": base_precision,
        "precision_gain": (
            explained_precision - base_precision
            if np.isfinite(explained_precision) and np.isfinite(base_precision) else np.nan
        ),
        "prediction_rule_consistency": float((predicted_alert == explained).mean()),
        "balanced_prediction_rule_consistency": (
            0.5 * (alert_support_rate + non_alert_no_evidence_rate)
            if np.isfinite(alert_support_rate) and np.isfinite(non_alert_no_evidence_rate)
            else np.nan
        ),
    }

## Results

In [ ]:
all_rules = truth.columns.tolist()
rows = [score_subset("full_rule_set", all_rules), score_subset("no_rules", [])]
rows += [score_subset(f"only:{rule}", [rule]) for rule in all_rules]
rows += [score_subset(f"without:{rule}", [item for item in all_rules if item != rule]) for rule in all_rules]
rows += [score_subset("full_rule_set_sensitivity", all_rules, value) for value in (0.50, 0.60, 0.70, 0.80)]
ablation = pd.DataFrame(rows)
display(ablation.round(4))
ablation.to_csv(output_dir / "ieee_rule_ablation.csv", index=False)
lineage_path = write_upstream_lineage(
    output_dir,
    "06_IEEE_CIS_Rule_Ablation",
    ["ieee_rule_ablation.csv"],
    PROJECT_ROOT / "configs/ieee_cis.yaml",
)
print({"upstream_lineage": str(lineage_path)})

## Takeaways

In [ ]:
full = ablation.query("ablation == 'full_rule_set'").iloc[0]
display(Markdown(
    f"- Full-set alert coverage: **{full['coverage_alerts']:.3f}**.\n"
    f"- Full-set precision gain: **{full['precision_gain']:.3f}**.\n"
    f"- Full-set explained-alert denominator: **{int(full['explained_alert_count'])}** alerts.\n"
    "- Leave-one-out and sensitivity rows are post-hoc locked-test diagnostics. "
    "No condition is selected as a new final rule set from these results."
))